## Project: Exploring NYC Public School SAT Results

**Author:** Jackie CW Vescio

The SAT is a standardized test with reading, math, and writing sections, each scored up to 800 points. It is intended to measure literacy, numeracy, and writing skills and plays an important role in college admissions. This project uses pandas to explore how New York City public schools performed on the SAT.

**Data:** The dataset, `schools.csv`, covers 375 NYC public schools across the five boroughs. It includes each school's borough, building code, average math, reading, and writing scores, and the percent of students tested. The file is not included in this repository. A small synthetic file, `sample_data/schools_sample.csv`, has the same columns so the code can be run, but its results will not match the findings in the README.

**Questions answered:**
1. Which schools have the best math results? (An average math score of at least 640, which is 80% of the 800 maximum.)
2. What are the top 10 schools by combined SAT score?
3. Which borough has the largest standard deviation in the combined SAT score?

**Extra checks:** I also tested whether one school drives Staten Island's spread, and I looked at `percent_tested` for the top 10 schools compared with the typical school. My written findings are in the README.

In [1]:
# Import packages
import pandas as pd
import numpy as np

# Read in the data
schools = pd.read_csv("schools.csv")

# Preview the data
schools.head()


,school_name,borough,building_code,average_math,average_reading,average_writing,percent_tested
0,"New Explorations into Science, Technology and ...",Manhattan,M022,657,601,601,NaN
1,Essex Street Academy,Manhattan,M445,395,411,387,78.9
2,Lower Manhattan Arts Academy,Manhattan,M445,418,428,415,65.1
3,High School for Dual Language and Asian Studies,Manhattan,M445,613,453,463,95.9
4,Henry Street School for International Studies,Manhattan,M056,410,406,381,59.7


In [2]:
# Display missing values

missing_df = schools.loc[schools['percent_tested'].isna()]

missing_df


,school_name,borough,building_code,average_math,average_reading,average_writing,percent_tested
0,"New Explorations into Science, Technology and ...",Manhattan,M022,657,601,601,NaN
19,Urban Assembly School of Business for Young Women,Manhattan,M282,418,420,417,NaN
38,High School of Hospitality Management,Manhattan,M535,451,433,424,NaN
57,Frederick Douglass Academy II Secondary School,Manhattan,M088,390,384,373,NaN
76,Jacqueline Kennedy Onassis High School,Manhattan,M486,418,422,415,NaN
95,Tottenville High School,Staten Island,R455,494,476,476,NaN
114,Bronx International High School,Bronx,X400,355,330,320,NaN
133,West Bronx Academy for the Future,Bronx,X435,404,383,378,NaN
152,Bronx High School for the Visual Arts,Bronx,X839,408,428,404,NaN
171,Kingsbridge International High School,Bronx,X430,366,311,310,NaN


In [3]:
# Print shape
schools.shape

# Print data types
schools.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 375 entries, 0 to 374
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   school_name      375 non-null    object 
 1   borough          375 non-null    object 
 2   building_code    375 non-null    object 
 3   average_math     375 non-null    int64  
 4   average_reading  375 non-null    int64  
 5   average_writing  375 non-null    int64  
 6   percent_tested   355 non-null    float64
dtypes: float64(1), int64(3), object(3)
memory usage: 20.6+ KB


In [4]:
# Which NYC schools have the best math results?

best_math_schools = schools.loc[
        schools['average_math'] >= 640,     # Only schools with a math score of at least 80% of 800
        ['school_name', 'average_math']
].sort_values(
    by='average_math', 
    ascending=False
)

best_math_schools

,school_name,average_math
88,Stuyvesant High School,754
170,Bronx High School of Science,714
93,Staten Island Technical High School,711
365,Queens High School for the Sciences at York Co...,701
68,"High School for Mathematics, Science, and Engi...",683
280,Brooklyn Technical High School,682
333,Townsend Harris High School,680
174,High School of American Studies at Lehman College,669
0,"New Explorations into Science, Technology and ...",657
45,Eleanor Roosevelt High School,641


In [5]:
# What are the top 10 performing schools based on the combined SAT score?

schools['total_SAT'] = (schools['average_math']+schools['average_reading']+schools['average_writing'])

top_10_schools = schools.loc[:,['school_name', 'total_SAT']].nlargest(10, 'total_SAT')

print(top_10_schools.shape)

top_10_schools

(10, 2)


,school_name,total_SAT
88,Stuyvesant High School,2144
93,Staten Island Technical High School,2041
170,Bronx High School of Science,2041
174,High School of American Studies at Lehman College,2013
333,Townsend Harris High School,1981
365,Queens High School for the Sciences at York Co...,1947
5,Bard High School Early College,1914
280,Brooklyn Technical High School,1896
45,Eleanor Roosevelt High School,1889
68,"High School for Mathematics, Science, and Engi...",1889


In [6]:
# Build DataFrame summary table to display borough, num_schools, average_SAT, and std_SAT 

borough_groups = schools.groupby('borough')

borough_summary_df = borough_groups.agg(
    num_schools = ('school_name', 'count'),
    average_SAT = ('total_SAT', 'mean'),
    std_SAT = ('total_SAT', 'std')
).reset_index()

borough_summary_df

,borough,num_schools,average_SAT,std_SAT
0,Bronx,98,1202.724490,150.393901
1,Brooklyn,109,1230.256881,154.868427
2,Manhattan,89,1340.134831,230.294140
3,Queens,69,1345.478261,195.252899
4,Staten Island,10,1439.000000,222.303596


In [7]:
# Which single borough has the largest standard deviation in the combined SAT score?

largest_std_dev = (borough_summary_df.nlargest(1, 'std_SAT').round(2)).reset_index(drop=True)

print(largest_std_dev.columns)
print(largest_std_dev.shape)

largest_std_dev

Index(['borough', 'num_schools', 'average_SAT', 'std_SAT'], dtype='object')
(1, 4)


,borough,num_schools,average_SAT,std_SAT
0,Manhattan,89,1340.13,230.29


In [8]:
# Robustness Check: 
# - Remove Staten Island Technical High School
# - Build DataFrame summary table to display borough, num_schools, average_SAT, and std_SAT 
# - Sort summary table by std_SAT
# Want to see if standard deviation spread comes mostly from one school.

# Remove one school (not the whole borough)
schools_no_si_tech = schools.loc[schools['school_name'] != 'Staten Island Technical High School']

# Build DataFrame summary table
borough_summary2_df = (
    schools_no_si_tech
    .groupby(['borough'])
    .agg(
     num_schools = ('school_name', 'count'),
     average_SAT = ('total_SAT', 'mean'),
     std_SAT = ('total_SAT', 'std'),
     )
)

# Sort summary table
borough_summary2_df = borough_summary2_df.sort_values(
     by=['std_SAT'],
     ascending=[False]
).reset_index()

# Display summary table
borough_summary2_df

,borough,num_schools,average_SAT,std_SAT
0,Manhattan,89,1340.134831,230.294140
1,Queens,69,1345.478261,195.252899
2,Brooklyn,109,1230.256881,154.868427
3,Bronx,98,1202.724490,150.393901
4,Staten Island,9,1372.111111,72.542133


#### Standard Deviation by NYC Borough
- The spread that nearly ties Manhattan with Staten Island is mainly from one school, Staten Island Technical High School.

In [9]:
# Check: What share of students took the SAT at the top 10 schools?
# Add percent_tested to the top 10 table to compare with the typical schools.

# Same top 10 as before, now including percent_tested
top_10_tested_df = schools.loc[:,['school_name', 'total_SAT', 'percent_tested']].nlargest(10, 'total_SAT')

print(top_10_tested_df.shape)

top_10_tested_df

(10, 3)


,school_name,total_SAT,percent_tested
88,Stuyvesant High School,2144,97.4
93,Staten Island Technical High School,2041,99.7
170,Bronx High School of Science,2041,97.0
174,High School of American Studies at Lehman College,2013,91.8
333,Townsend Harris High School,1981,97.1
365,Queens High School for the Sciences at York Co...,1947,97.9
5,Bard High School Early College,1914,70.8
280,Brooklyn Technical High School,1896,95.5
45,Eleanor Roosevelt High School,1889,86.0
68,"High School for Mathematics, Science, and Engi...",1889,92.6


In [10]:
# Find the smallest and largest percent_tested among the top 10 schools now including percent_tested.

# Highest percent tested
highest_tested_df = top_10_tested_df.nlargest(1, 'percent_tested')

# Lowest percent tested
lowest_tested_df = top_10_tested_df.nsmallest(1, 'percent_tested')

# Combine highest and lowest percent tested
highest_lowest_tested_df = pd.concat([highest_tested_df, lowest_tested_df]).reset_index(drop=True)

# Display results
highest_lowest_tested_df

,school_name,total_SAT,percent_tested
0,Staten Island Technical High School,2041,99.7
1,Bard High School Early College,1914,70.8


In [11]:
# Display the "typical school"

median_tested = schools['percent_tested'].median()

median_df = pd.DataFrame({
    'measure': ['Median Percent Tested, All Schools'],
    'percent_tested': [median_tested.round(1)]
})

median_df

,measure,percent_tested
0,"Median Percent Tested, All Schools",64.8


#### Share of Students Tested
- top-10 range (about 71% to 100%) 
- typical school (about 65%)

Note: Who takes the test may affect the averages